In [2]:
import bldw
import numpy as np
import blimpy as bl
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob
import setigen as stg
from astropy import units as u
%matplotlib inline

plt.rcParams["font.family"] = "serif"
plt.rcParams["mathtext.fontset"] = "dejavuserif"

In [12]:
dirstring = '/datax/scratch/benjb/bl_nearby_stars/bliss_final_100_search_063026/*.dat'

ivec = []

count_empty = 0
for i, file in enumerate(glob.iglob(dirstring)):
    if i%100000 == 0:
        print(i)
    size_bytes = os.path.getsize(file)
    if size_bytes==0:
        print('found empty')
        count_empty += 1
    index = os.path.basename(file).split('_')[0]
    ivec.append(index)

# filename starts with {index}_{h5idx}_{k}

# k is the coarse channel number

0
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
100000
found empty
found empty
found empty
found e

In [13]:
print(count_empty/len(ivec))
print(count_empty, len(ivec))

0.003737085073642559
1632 436704


In [ ]:
# dirstring = '/datax/scratch/benjb/bl_nearby_stars/bliss_dats_pooled_090825/*.dat'

# ivec2 = []

# for i, file in enumerate(glob.iglob(dirstring)):
#     if i%100000 == 0:
#         print(i)
#     index = os.path.basename(file).split('_')[0]
#     ivec2.append(index)

# # filename starts with {index}_{h5idx}_{k}

# # k is the coarse channel number

0
100000


In [14]:
u = np.sort(np.unique(ivec).astype(int))
# u2 = np.sort(np.unique(ivec2).astype(int))

In [6]:
# np.savez('/datax/scratch/benjb/bl_nearby_stars/blpc2_spliced_files_idx_092925.npz', u)

In [15]:
print(len(u))

122


In [8]:
print(list(u))
#print(list(u2))

[6231, 6279, 6280, 12366, 13167, 20394, 20423, 21660, 22080, 22420, 23565, 24835, 24836, 24837, 24838, 24839, 24840, 24841, 24842, 24843, 24844, 24845, 24846, 24847, 24848, 24849, 24850, 24880, 24881, 24882, 24883, 24884, 24885, 24886, 24887, 24888, 24889, 24890, 24891, 24892, 24893, 24894, 24895, 24896, 24897, 24898, 24923, 24954, 24967, 25013, 25014, 25015, 25016, 25017, 25018, 25019, 25020, 25021, 25458, 25459, 25490, 25491, 25521, 25522, 25553, 25554, 25555, 25608, 25617, 25618, 25619, 25620, 25658, 25659, 25660, 25713, 25714, 25715, 25716, 25717, 25718, 25719, 25720, 25721, 25722, 25723, 25724, 25748, 25749, 25750, 25751, 25775, 25776, 25777, 25778, 25779, 25833, 25834, 25835, 25836, 25860, 25861, 25862, 25863, 25887, 25917, 25918, 25919, 25993, 27007, 28280, 28284, 28671, 39112, 39113]


In [16]:
def combine_hits(list_of_dats):
    cadvec = []
    scanvec = []
    ccvec = []
    for dat in list_of_dats:
        spl = os.path.basename(dat).split('_')
        cadvec.append(spl[0])
        scanvec.append(spl[1])
        ccvec.append(spl[2])
    ccvec = np.array(ccvec).astype('int')
    cadi = np.unique(cadvec)
    scani = np.unique(scanvec)
    if len(cadi) > 1:
        print(f'Too many cadence indices: {cadi}')
    if len(scani) > 1:
        print(f'Too many scan indices: {scani}')
    if np.isin(np.arange(len(ccvec)), ccvec).all():
        print(f'{cadi}: All coarse channels accounted for!')
    else:
        print(f'{cadi}: Missing coarse channels:', ccvec[~np.isin(np.arange(len(ccvec)), ccvec)])
    ### write header
    size_list = np.array([os.path.getsize(file) for file in list_of_dats])
    print(size_list)
    print(len(size_list))
    if len(np.where(size_list > 0)[0]) > 0:
        j = np.where(size_list > 0)[0][0]
        with open(list_of_dats[j]) as f:
            print(list_of_dats[j])
            header = [next(f) for _ in range(9)]
        with open(f'/datax/scratch/benjb/bl_nearby_stars/respliced_remaining_dats_071326/{os.path.basename(list_of_dats[0])}', 'w') as f:
            for hline in header:
                f.write(hline)
        ### write hits
        for j in range(len(list_of_dats)):
            with open(list_of_dats[j]) as f:
                all_lines = f.readlines()
                if len(all_lines) > 0:
                    lines = all_lines[9:]
                else:
                    lines = []
            with open(f'/datax/scratch/benjb/bl_nearby_stars/respliced_remaining_dats_071326/{os.path.basename(list_of_dats[0])}', 'a') as f:
                for line in lines:
                    f.write(line)

In [17]:
for i, ui in enumerate(u):
    #if ui < 39113:
    #    continue
    cadfiles = glob.glob(f'/datax/scratch/benjb/bl_nearby_stars/bliss_final_100_search_063026/{ui}_*.dat')
    #print(np.sort(np.array(cadfiles)))
    allmjds = [os.path.basename(file).split('guppi_')[1][:11] for file in cadfiles]
    mjdvec = np.unique(allmjds)
    print(mjdvec)
    ulen = len(str(ui)) # number of digits in index
    #print(len(files))
    for j in range(6):
        print(ui, j)
        if j+1 > len(mjdvec):
            continue
        # go through all files and put the ones from, e.g., scan 1 into a separate list
        scanfiles = []
        for cadfile in cadfiles:
            mjd_check = os.path.basename(cadfile).split('guppi_')[1][:11]
            if mjd_check == mjdvec[j]:
            #if int(os.path.basename(cadfile)[ulen+1])-1 == j:
                scanfiles.append(cadfile)
        scanfiles = np.sort(np.array(scanfiles))
        # write new function to put all hits into new dat file
        #print(scanfiles)
        combine_hits(scanfiles)
        #print(scanfiles[-1])
        #break
    #break

['58231_58600']
5412 0
['5412']: All coarse channels accounted for!
[0 0 0 ... 0 0 0]
1452
5412 1
5412 2
5412 3
5412 4
5412 5
['58265_43116' '58265_43449' '58265_43783' '58265_44116' '58265_44451'
 '58265_44784']
6231 0
['6231']: All coarse channels accounted for!
[1526  670  670 ...  670  670  874]
1664
/datax/scratch/benjb/bl_nearby_stars/bliss_final_100_search_063026/6231_1_0_spliced_blc00010203040506o7o0111213141516o7o0212223242526o7o031323334353637_guppi_58265_43116_HIP116495_0021.gpuspec.0000_nosig_nosk_SNR_20_L1_30.dat
6231 1
['6231']: All coarse channels accounted for!
[1521  669  669 ...  669  669  669]
1664
/datax/scratch/benjb/bl_nearby_stars/bliss_final_100_search_063026/6231_2_0_spliced_blc00010203040506o7o0111213141516o7o0212223242526o7o031323334353637_guppi_58265_43449_HIP115779_0022.gpuspec.0000_nosig_nosk_SNR_20_L1_30.dat
6231 2
['6231']: All coarse channels accounted for!
[1526  670  670 ...  670  670  670]
1664
/datax/scratch/benjb/bl_nearby_stars/bliss_final_100_sea

In [ ]:
# check unspliced dats -- needs to be done separately for each blpc{0,1,2,3}
# check spliced dats -- currently doing for blpc2, needs to be done for blpc{0,1,3}
# merge all spliced dats -- currently doing for blpc2, needs to be done for blpc{0,1,3}
# pool dats together on a single blpc machine (likely blpc2)
# update CSV
# read off hits and make plots!
# start FindEvent

# figure out ad hoc plotting code for large turboSETI event counts (plot_large_dat_files.ipynb)